# TP 1 — Partie 2 : nettoyage et validation en couche Silver

**Objectif :** transformer les tables Bronze en tables **propres, typées et fiables**.

En Silver, chaque table :
- a des types corrects (dates, nombres) et des noms de colonnes clairs ;
- ne contient plus de doublons sur sa clé ;
- respecte des règles de qualité explicites. Les lignes qui violent une règle **bloquante** ne sont pas supprimées en silence : elles sont isolées dans une table de **quarantaine**.

> **⚠️ Un seul notebook actif à la fois.** Chaque notebook ouvert garde sa propre session Spark (jusqu'à 2 Go de mémoire). Avant de commencer, arrêtez le noyau des notebooks précédents : panneau *Running Terminals and Kernels* (barre de gauche) > *Shut Down All*. Les tables restent dans le Lakehouse.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("tp1-silver").getOrCreate()


def controle(df, nom, cle):
    """Affiche le nombre de lignes, de doublons sur la clé et de valeurs nulles par colonne."""
    colonnes = [c for c in df.columns if not c.startswith("_") and c != "date_ingestion"]
    total = df.count()
    doublons = total - df.dropDuplicates(cle).count()
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colonnes]).first().asDict()
    nulls = {c: n for c, n in nulls.items() if n}
    print(f"{nom:28s} {total:>7} lignes | doublons sur {cle} : {doublons} | nulls : {nulls or 'aucun'}")


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")

## Étape 1 — Profiler la couche Bronze

Avant de nettoyer, on mesure. Exécutez le contrôle sur les tables Bronze.

In [ ]:
CLES = {
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "products": ["product_id"],
    "category_translation": ["product_category_name"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
}
for table, cle in CLES.items():
    controle(spark.table(f"lakehouse.bronze.{table}"), f"bronze.{table}", cle)

> **❓ Question 1.** Relevez les tables et colonnes qui posent problème. Pour chacune, est-ce une **erreur** de la source ou une **situation normale** du métier ? (Exemple : une commande non livrée n'a pas de date de livraison.)

## Étape 2 — Exemple guidé : `silver.customers`

Lisez attentivement cette transformation : vous appliquerez la même démarche aux autres tables.

In [ ]:
customers = (spark.table("lakehouse.bronze.customers")
    .select(
        F.trim("customer_id").alias("customer_id"),
        F.trim("customer_unique_id").alias("customer_unique_id"),
        F.lpad(F.trim("customer_zip_code_prefix"), 5, "0").alias("customer_zip_code_prefix"),
        F.initcap(F.trim("customer_city")).alias("customer_city"),
        F.upper(F.trim("customer_state")).alias("customer_state"),
    )
    .dropDuplicates(["customer_id"]))

customers.writeTo("lakehouse.silver.customers").using("iceberg").createOrReplace()
controle(spark.table("lakehouse.silver.customers"), "silver.customers", ["customer_id"])
spark.table("lakehouse.silver.customers").show(3)

## Étape 3 — `silver.sellers`

La table des vendeurs suit exactement la même démarche : le code est fourni. Comparez-le à celui des clients, puis exécutez-le.

In [ ]:
sellers = (spark.table("lakehouse.bronze.sellers")
    .select(
        F.trim("seller_id").alias("seller_id"),
        F.lpad(F.trim("seller_zip_code_prefix"), 5, "0").alias("seller_zip_code_prefix"),
        F.initcap(F.trim("seller_city")).alias("seller_city"),
        F.upper(F.trim("seller_state")).alias("seller_state"),
    )
    .dropDuplicates(["seller_id"]))

sellers.writeTo("lakehouse.silver.sellers").using("iceberg").createOrReplace()
controle(spark.table("lakehouse.silver.sellers"), "silver.sellers", ["seller_id"])

## Étape 4 — `silver.products`

La table des produits demande plus de travail. Créez `lakehouse.silver.products` :
1. **Renommez** les colonnes mal orthographiées dans la source : `product_name_lenght` → `product_name_length`, `product_description_lenght` → `product_description_length`.
2. **Typez** : `product_photos_qty` et les longueurs en entier, `product_weight_g` et les trois dimensions (`product_length_cm`, `product_height_cm`, `product_width_cm`) en `double`.
3. Un poids égal à `0` est impossible : remplacez-le par `null`.
4. Remplacez une catégorie manquante par `"sem_categoria"`.
5. Ajoutez la colonne `product_category_en` grâce à la table `bronze.category_translation` (jointure gauche). Si aucune traduction n'existe, gardez le nom portugais.

In [ ]:
traduction = spark.table("lakehouse.bronze.category_translation").select(
    "product_category_name", "product_category_name_english")

products = a_completer("products")  # À COMPLÉTER

products.writeTo("lakehouse.silver.products").using("iceberg").createOrReplace()
controle(spark.table("lakehouse.silver.products"), "silver.products", ["product_id"])

> **❓ Question 2.** Quelles catégories n'ont pas de traduction ? Écrivez la requête qui les trouve.

## Étape 5 — `silver.orders` et sa quarantaine

Les commandes viennent de PostgreSQL : elles sont déjà typées. Le travail porte sur les **doublons** et les **règles métier**.

Créez un DataFrame `orders` qui :
1. **garde une seule version par commande**, la plus récente selon `updated_at` (fenêtre `Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())` et `F.row_number()`). La table Bronze n'a pas encore de doublons, mais elle en aura après l'ingestion incrémentale de la partie 3 ;
2. ajoute `delai_livraison_jours` (jours entre l'achat et la livraison au client), `delai_estime_jours` (jours entre l'achat et la date de livraison estimée) et `en_retard` (`true` si la livraison a eu lieu après la date estimée, `null` si la commande n'est pas livrée) ;
3. ajoute deux indicateurs de qualité :
   - `dq_livree_sans_date` : statut `delivered` mais aucune date de livraison (**règle bloquante**) ;
   - `dq_transporteur_avant_approbation` : remise au transporteur avant l'approbation du paiement (**simple alerte**).

Ensuite, écrivez les lignes bloquées dans `lakehouse.silver.quarantaine_orders` et les autres dans `lakehouse.silver.orders`.

In [ ]:
dernier = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

orders = a_completer("orders (dédoublonnage + colonnes calculées + indicateurs dq_)")  # À COMPLÉTER

# À COMPLÉTER : écriture de la quarantaine puis de silver.orders

controle(spark.table("lakehouse.silver.orders"), "silver.orders", ["order_id"])
print("en quarantaine :", spark.table("lakehouse.silver.quarantaine_orders").count())

> **❓ Question 3.** Combien de commandes ont été remises au transporteur avant l'approbation du paiement ? Pourquoi est-ce une **alerte** et non une règle bloquante ?

## Étape 6 — Articles et paiements

La table des articles est fournie : elle dédoublonne sur la clé, convertit les montants en `double` et retire les colonnes techniques.
Créez sur le même modèle `lakehouse.silver.order_payments` (montant : `payment_value`), et isolez en quarantaine (`lakehouse.silver.quarantaine_payments`) les lignes dont le `payment_type` vaut `not_defined`.

In [ ]:
items = (spark.table("lakehouse.bronze.order_items")
    .dropDuplicates(CLES["order_items"])
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
    .drop("_source", "_ingere_le"))
items.writeTo("lakehouse.silver.order_items").using("iceberg").createOrReplace()

payments = a_completer("payments (sur le modèle des articles, plus la quarantaine)")  # À COMPLÉTER

controle(spark.table("lakehouse.silver.order_items"), "silver.order_items", CLES["order_items"])
controle(spark.table("lakehouse.silver.order_payments"), "silver.order_payments", CLES["order_payments"])

## Étape 7 — Intégrité référentielle

Une jointure ne doit pas perdre de lignes en silence. Vérifiez avec des **anti-jointures** (`how="left_anti"`) :
- les articles dont le produit n'existe pas dans `silver.products` ;
- les commandes sans aucun article ;
- les commandes dont le client n'existe pas dans `silver.customers`.

In [ ]:
# À COMPLÉTER : trois anti-jointures, affichez le nombre de lignes de chacune

> **❓ Question 4.** Des commandes n'ont aucun article. Regardez leur statut : quelle hypothèse pouvez-vous faire ? Comment faudra-t-il les traiter dans le calcul du chiffre d'affaires ?

## Étape 8 — Bilan

Le tableau ci-dessous compare Bronze et Silver. Chaque écart doit pouvoir être **expliqué** par une règle : c'est ce qu'on appelle la traçabilité.

In [ ]:
bilan = []
for table in ["customers", "sellers", "products", "orders", "order_items", "order_payments"]:
    bronze = spark.table(f"lakehouse.bronze.{table}").count()
    silver = spark.table(f"lakehouse.silver.{table}").count()
    bilan.append((table, bronze, silver, bronze - silver))
spark.createDataFrame(bilan, ["table", "bronze", "silver", "ecart"]).show()
spark.sql("SHOW TABLES IN lakehouse.silver").show()